In [ ]:
import pandas as pd

train = pd.read_csv('ChurnZero_dataset_v1.csv')
test = pd.read_csv('ChurnZero_test_v1.csv')

print("Training data shape:", train.shape)
print("Test data shape:", test.shape)
print(train.head())

Training data shape: (8101, 98)
Test data shape: (2026, 97)
   customer_id  churn  age  gender marital_status education_level  \
0       133001      0   36  Female        Married         Unknown   
1       133002      1   44    Male         Single     High School   
2       133003      0   46    Male        Married     High School   
3       133004      0   36  Female        Married        Graduate   
4       133005      0   50    Male         Single        Graduate   

   dependent_count occupation_type  annual_income   income_band  ...  \
0                0        Salaried          22256           Low  ...   
1                3       Homemaker          66481        Middle  ...   
2                2        Salaried          98955  Upper-Middle  ...   
3                2         Retired          26735           Low  ...   
4                1        Salaried         165387          High  ...   

  discount_or_fee_waiver_received competitor_bank_offer_awareness  \
0                      

In [ ]:
print("Total customers:", len(train))
print("\nChurn distribution:")
print(train['churn'].value_counts())
print("\nChurn percentage:", round(train['churn'].mean()*100, 2), "%")

Total customers: 8101

Churn distribution:
churn
0    6799
1    1302
Name: count, dtype: int64

Churn percentage: 16.07 %


In [ ]:
print("Missing values in training data:")
print(train.isnull().sum()[train.isnull().sum() > 0])

Missing values in training data:
app_rating_given    4541
dtype: int64


In [ ]:
print("All columns:")
for col in train.columns:
    print(col)

All columns:
customer_id
churn
age
gender
marital_status
education_level
dependent_count
occupation_type
annual_income
income_band
income_category
city_tier
region
customer_segment
tenure_months
onboarding_channel
relationship_type
number_of_products
primary_account_type
card_category
customer_lifetime_value
loyalty_program_member
referral_count
last_contacted_days
relationship_manager_assigned
avg_monthly_balance
current_balance
balance_decline_percentage
monthly_transaction_count
monthly_transaction_value
cash_withdrawal_count
upi_transaction_count
debit_card_transaction_count
net_banking_transaction_count
account_inactive_days
total_trans_amt
total_trans_count
total_amt_chng_q4_q1
total_ct_chng_q4_q1
avg_open_to_buy
total_revolving_bal
savings_account_flag
current_account_flag
credit_card_flag
personal_loan_flag
home_loan_flag
auto_loan_flag
fixed_deposit_flag
investment_product_flag
insurance_product_flag
demat_account_flag
credit_card_limit
credit_card_spend
credit_utilization_rat

In [6]:
# Fill missing values
train['app_rating_given'] = train['app_rating_given'].fillna(train['app_rating_given'].median())
test['app_rating_given'] = test['app_rating_given'].fillna(train['app_rating_given'].median())

# Separate features and target
X = train.drop(columns=['churn'])
y = train['churn']

# Find text columns and convert to numbers
cat_cols = X.select_dtypes(include='object').columns.tolist()
print("Text columns to encode:", cat_cols)

from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

for col in cat_cols:
    X[col] = le.fit_transform(X[col].astype(str))
        if col in test.columns:
                test[col] = le.transform(test[col].astype(str))

                print("✅ Data cleaned successfully!")
                print("Shape:", X.shape)

IndentationError: unexpected indent (1617648984.py, line 18)

In [7]:
# Fill missing values
train['app_rating_given'] = train['app_rating_given'].fillna(train['app_rating_given'].median())
test['app_rating_given'] = test['app_rating_given'].fillna(train['app_rating_given'].median())

# Separate features and target
X = train.drop(columns=['churn'])
y = train['churn']

# Find text columns and convert to numbers
from sklearn.preprocessing import LabelEncoder

cat_cols = X.select_dtypes(include='object').columns.tolist()
print("Text columns to encode:", cat_cols)

le = LabelEncoder()

for col in cat_cols:
    X[col] = le.fit_transform(X[col].astype(str))
    if col in test.columns:
        test[col] = le.transform(test[col].astype(str))

print("Data cleaned successfully!")
print("Shape:", X.shape)

Text columns to encode: ['gender', 'marital_status', 'education_level', 'occupation_type', 'income_band', 'income_category', 'city_tier', 'region', 'customer_segment', 'onboarding_channel', 'relationship_type', 'primary_account_type', 'card_category', 'competitor_bank_offer_awareness', 'customer_feedback_sentiment']
Data cleaned successfully!
Shape: (8101, 97)


In [8]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score

# Split data for validation
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training size:", X_train.shape)
print("Validation size:", X_val.shape)

# Train XGBoost model
model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    scale_pos_weight=5,
    random_state=42,
    eval_metric='logloss'
)

model.fit(X_train, y_train)
print("✅ Model trained successfully!")

# Evaluate
y_pred = model.predict(X_val)
y_prob = model.predict_proba(X_val)[:,1]

print("\n--- Results ---")
print(classification_report(y_val, y_pred))
print("ROC-AUC Score:", round(roc_auc_score(y_val, y_prob), 4))

Training size: (6480, 97)
Validation size: (1621, 97)
✅ Model trained successfully!

--- Results ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1360
           1       1.00      0.99      0.99       261

    accuracy                           1.00      1621
   macro avg       1.00      0.99      1.00      1621
weighted avg       1.00      1.00      1.00      1621

ROC-AUC Score: 1.0


In [9]:
# Check if any column is too perfectly correlated with churn
correlations = X.corrwith(y).abs().sort_values(ascending=False)
print("Top 10 most correlated features:")
print(correlations.head(10))

Top 10 most correlated features:
total_digital_logins           0.507769
unresolved_complaint_count     0.494274
customer_feedback_sentiment    0.458200
balance_decline_percentage     0.455938
complaint_resolution_time      0.448285
mobile_app_login_count         0.427283
cash_withdrawal_count          0.425809
email_open_rate                0.411138
last_campaign_response_days    0.404755
emi_payment_delay_count        0.403164
dtype: float64


/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


In [10]:
# Retrain with overfitting prevention
model = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=5,
    min_child_weight=5,
    random_state=42,
    eval_metric='logloss'
)

model.fit(X_train, y_train)

y_pred = model.predict(X_val)
y_prob = model.predict_proba(X_val)[:,1]

print("--- Results ---")
print(classification_report(y_val, y_pred))
print("ROC-AUC Score:", round(roc_auc_score(y_val, y_prob), 4))


--- Results ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1360
           1       1.00      0.99      0.99       261

    accuracy                           1.00      1621
   macro avg       1.00      0.99      1.00      1621
weighted avg       1.00      1.00      1.00      1621

ROC-AUC Score: 1.0


In [11]:
# Check if customer_id or any ID column exists
print("First 5 columns:", X.columns[:5].tolist())
print("\nLast 5 columns:", X.columns[-5:].tolist())

# Check columns with suspiciously high unique values
for col in X.columns:
    unique_ratio = X[col].nunique() / len(X)
    if unique_ratio > 0.9:
        print(f"Suspicious column: {col} — {X[col].nunique()} unique values")

First 5 columns: ['customer_id', 'age', 'gender', 'marital_status', 'education_level']

Last 5 columns: ['credit_utilization_6m_avg', 'avg_quarterly_balance', 'total_digital_logins', 'debt_to_income_ratio', 'digital_engagement_index']
Suspicious column: customer_id — 8101 unique values
Suspicious column: annual_income — 7421 unique values
Suspicious column: customer_lifetime_value — 8092 unique values
Suspicious column: avg_quarterly_balance — 7711 unique values


In [12]:
# Check if churn column accidentally stayed in X
print("'churn' in X columns?", 'churn' in X.columns)
print("'Exited' in X columns?", 'Exited' in X.columns)

# Check columns that are almost identical to churn
for col in X.columns:
    if X[col].nunique() == 2:
        corr = abs(X[col].corr(y))
        if corr > 0.8:
            print(f"HIGH CORRELATION: {col} = {corr:.3f}")

'churn' in X columns? False
'Exited' in X columns? False


In [13]:
# Drop columns that cause memorization
cols_to_drop = ['customer_id', 'annual_income',
                'customer_lifetime_value', 'avg_quarterly_balance']

X_clean = X.drop(columns=cols_to_drop, errors='ignore')
test_clean = test.drop(columns=cols_to_drop, errors='ignore')

print("New shape:", X_clean.shape)

# Split again
X_train2, X_val2, y_train2, y_val2 = train_test_split(
    X_clean, y, test_size=0.2, random_state=42, stratify=y
)

# Retrain
model2 = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=5,
    min_child_weight=5,
    random_state=42,
    eval_metric='logloss'
)

model2.fit(X_train2, y_train2)

y_pred2 = model2.predict(X_val2)
y_prob2 = model2.predict_proba(X_val2)[:,1]

print("\n--- Results ---")
print(classification_report(y_val2, y_pred2))
print("ROC-AUC Score:", round(roc_auc_score(y_val2, y_prob2), 4))

New shape: (8101, 93)

--- Results ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1360
           1       1.00      0.99      0.99       261

    accuracy                           1.00      1621
   macro avg       1.00      0.99      1.00      1621
weighted avg       1.00      1.00      1.00      1621

ROC-AUC Score: 0.9999


In [14]:
# Generate final predictions on test data
test_predictions = model2.predict(test_clean)
test_probabilities = model2.predict_proba(test_clean)[:,1]

# Check if test has customer_id
if 'customer_id' in test.columns:
    submission = pd.DataFrame({
        'customer_id': test['customer_id'],
        'churn': test_predictions
    })
else:
    submission = pd.DataFrame({
        'churn': test_predictions
    })

print("Submission shape:", submission.shape)
print("\nPrediction distribution:")
print(submission['churn'].value_counts())
print("\nFirst 5 rows:")
print(submission.head())

# Save to CSV
submission.to_csv('predictions.csv', index=False)
print("\n✅ predictions.csv saved successfully!")

Submission shape: (2026, 2)

Prediction distribution:
churn
0    1700
1     326
Name: count, dtype: int64

First 5 rows:
   customer_id  churn
0    767114958      0
1    708123033      0
2    715424283      0
3    717865008      0
4    710188308      0

✅ predictions.csv saved successfully!


In [15]:
from google.colab import files
files.download('predictions.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>